# SEVIRI SSL: training performance

This notebook measures the time spent loading satellite clips, preparing batches, executing the model, computing gradients, updating weights and evaluating validation observations. It refers to the 150-epoch experiment series in `results/ssl_scaling_20261025` and to the four objectives described in the [project specification](../handoff_codex_ssl_seviri.md): masked video modeling (MVM), direct future-frame prediction (Future), their joint objective (Joint), and advection-residual prediction (Residual). 

The initial performance tests use separate randomly initialized models and separate output directories. The full-epoch adoption tests below also include the checkpoint continuation of MVM on 8,192 clips. The scientific MVM run completed 150 epochs before these tests started. All measurements below concern this workspace and its assigned GPU partition.

The measurements identify CPU input preparation, memory copies and competition for CPU execution time as the main throughput constraint. The compute path processes substantially more clips when inputs already reside on the GPU. Larger GPU batches provide an additional benefit that depends on the training objective; the sustained pipeline measurements below quantify that difference.

The follow-up sections **Decoded frames and batch delivery** and **GPU activity with reduced tracing overhead** report the faster input pipeline and its comparison with GPU-resident execution.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "handoff_codex_ssl_seviri.md").exists())
REVIEW = ROOT / "results/ssl_scaling_20261025/performance_review"
FIGURES = REVIEW / "figures"
FIGURES.mkdir(exist_ok=True)

def read(name):
    return json.loads((REVIEW / name / "result.json").read_text())

def show_table(rows, formats=None):
    frame = pd.DataFrame(rows)
    display(frame.style.format(formats or {}, na_rep="—").hide(axis="index"))
    return frame

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
OBJECTIVES = {"mvm": "MVM", "future": "Future", "joint": "Joint", "residual": "Residual", "probe": "Probe"}


## Hardware and measurement conditions

The workspace exposes 11 CPU cores, approximately 61 GiB of RAM and an NVIDIA H200X-2-35C virtual GPU with a MIG 2g.35gb partition. The assigned partition contains 32 streaming multiprocessors and approximately 32 GiB of usable GPU memory. The training precision is bfloat16.

A **GPU batch** is the group of clips processed in one forward and backward pass. An **I/O batch** is the group returned by one DataLoader task. The original execution configuration uses GPU batches of 16 clips, I/O batches of 2 clips and eight worker processes. Two successive GPU batches contribute gradients to each weight update, giving an **effective batch of 32 clips**. The main process joins eight I/O batches into one pinned-memory tensor before transferring it to the GPU. Each clip occupies approximately 21 MiB as a float32 tensor, so a GPU batch of 16 requires approximately 343 MiB of host input storage.

The steady pipeline comparisons below each process 2,048 training clips after warming up the loader and CUDA operations. Additional unread clips keep worker processes active throughout the measurement. All configurations read the same deterministic ordering from the 4,096-clip manifest and use its stored global normalization. GPU batches of 32 use one update per batch to retain an effective batch of 32. The timed interval covers training batches after warmup; startup and final checkpoint serialization are measured separately.

The initial short trials are preserved in the result directory. Their larger I/O batches allowed workers to finish reading much of the input before the timed interval ended. The tables below use the subsequent steady measurements to compare sustained throughput.

## Measured experiment series duration

These rows summarize the last 20 epochs of the completed scientific runs. Training time includes the entire training sample and batch preparation. Validation time includes all 512 validation clips, persistence and advection baselines, forecast metrics, variability groups and the forecast figure. Each epoch also saves a recoverable checkpoint.

In [ ]:
rows = []
for path in sorted((ROOT / "results/ssl_scaling_20261025").glob("n*/*/loss_epochs.csv")):
    progress = json.loads((path.parent / "progress.json").read_text())
    if progress["status"] != "completed":
        continue
    epochs = pd.read_csv(path).tail(20)
    rows.append({"Training clips": int(path.parent.parent.name[1:]), "Objective": OBJECTIVES.get(path.parent.name, path.parent.name),
                 "Completed epochs": progress["completed_epochs"], "Training (s/epoch)": epochs.train_seconds.mean(),
                 "Validation (s/epoch)": epochs.validation_seconds.mean(), "Training (clips/s)": epochs.clips_per_second.mean()})
series_timings = show_table(rows, {"Training (s/epoch)": "{:.1f}", "Validation (s/epoch)": "{:.1f}", "Training (clips/s)": "{:.0f}"})


## Time spent in each training operation

The following table reports the mean elapsed time observed by the main Python process for one GPU batch of 16 clips, with the experiment series's I/O batch size of 2. The durations follow the actual sequence of operations, including any synchronization encountered inside a call. Gradient checks and optimizer updates run every second GPU batch; their displayed durations are averaged over all GPU batches.

CUDA launches are asynchronous. A Python call can return while its GPU work continues, and a later scalar read can wait for that work. These elapsed intervals describe where the training process spends its time. The activity traces below separately measure the intervals occupied by GPU kernels and transfers. Data loading runs concurrently in worker processes, so its complete CPU cost also contributes to contention while the main process executes the model.

In [ ]:
stages = [
    ("Wait for DataLoader", "loader_wait_ms"), ("Assemble pinned batch", "assembly_ms"),
    ("Transfer and select temporal window", "host_to_device_and_window"),
    ("Prepare masks / advection", "mask_and_advection"), ("Forward pass and loss", "forward_and_loss"),
    ("Check finite loss", "finite_loss_check"), ("Backward pass", "backward"),
    ("Check gradient norm", "gradient_norm_check"), ("Update weights", "optimizer_step"),
    ("Read scalar metrics", "scalar_metrics"), ("Write batch CSV", "step_log_csv")]
rows = []
for label, key in stages:
    row = {"Operation": label}
    for objective in ["mvm", "future"]:
        r = read(f"steady_{objective}_b16_io2")
        if key in r["batch_io"]:
            value = r["batch_io"][key]["mean_ms"]
        else:
            timing = r["timings"]["host_intervals"][key]
            value = timing["mean_ms"] * timing["samples"] / r["batch_wall"]["samples"]
        row[f"{OBJECTIVES[objective]} (ms/batch)"] = value
    rows.append(row)
operations = show_table(rows, {"MVM (ms/batch)": "{:.1f}", "Future (ms/batch)": "{:.1f}"})
show_table([{"Objective": OBJECTIVES[o], "Total (ms/batch)": read(f"steady_{o}_b16_io2")["batch_wall"]["mean_ms"],
             "CPU cores in use, mean": read(f"steady_{o}_b16_io2")["resources"]["cpu_seconds"] / read(f"steady_{o}_b16_io2")["wall_seconds"],
             "Physical reads (MiB)": read(f"steady_{o}_b16_io2")["resources"]["physical_read_bytes"] / 2**20}
            for o in ["mvm", "future"]],
           {"Total (ms/batch)": "{:.1f}", "CPU cores in use, mean": "{:.1f}", "Physical reads (MiB)": "{:.0f}"});


## Reading one clip and saving a checkpoint

The single-reader measurements use 31 clips after the first access has initialized the reader. “Read, validate and orient” includes Zarr access and decoding, temporal and channel checks, cropping, coordinate-based orientation and finite-value checks. Normalization applies the stored global training mean and standard deviation. The tensor view shares the contiguous NumPy storage. These per-clip durations were measured without the competing training workers and therefore describe a different concurrency condition from the main-process table.

The physical-read counters measure bytes requested from storage by the benchmark processes. Zero physical reads during a timed interval means that the filesystem cache supplied the requested content; Zarr decoding and memory copies still require CPU work. The cached 32-clip diagnostic and GPU-resident tests reuse real inputs to isolate parts of the pipeline. A persistent training cache would require a separate validation of the full dataset and its storage cost.

The native Zarr blocks contain one time step, all seven channels and a 224 × 224 pixel tile, stored as float16 with Blosc Zstd compression. A 16-frame clip requests 16, 32 or 64 blocks depending on the alignment of its spatial crop. The 4,096-clip manifest requests approximately 35 blocks per clip on average. Spatially offset crops therefore decode additional pixels around the selected crop.

In [ ]:
r = read("io_baseline")
show_table([{"Operation": name, "Time (ms/clip)": r["single_reader_warm"][key]["mean_ms"]}
            for name,key in [("Read, validate and orient", "read_validate_orient_ms"), ("Global normalization", "normalize_ms"), ("Tensor view", "tensor_view_ms")]],
           {"Time (ms/clip)": lambda value: "<0.1" if value < 0.1 else f"{value:.1f}"});
show_table([{"Objective": OBJECTIVES[o], "Startup + warmup (s)": read(f"steady_{o}_b16_io2")["startup_and_warmup_seconds"],
             "Save checkpoint (s)": read(f"steady_{o}_b16_io2")["checkpoint_seconds"],
             "Checkpoint (MiB)": read(f"steady_{o}_b16_io2")["checkpoint_bytes"] / 2**20}
            for o in ["mvm", "future"]],
           {"Startup + warmup (s)": "{:.1f}", "Save checkpoint (s)": "{:.1f}", "Checkpoint (MiB)": "{:.0f}"});


### Input preparation control

This control keeps 32 real, globally normalized clips in shared CPU memory and repeatedly reads them through the same DataLoader configuration as the experiment series baseline: GPU batch 16, I/O batch 2, eight workers and gradient accumulation over two batches. It measures 128 batches after 32 warmup batches, while preserving input collation, worker-to-parent communication, pinned-memory assembly, GPU transfer and training. Comparing it with the baseline isolates the combined contribution of Zarr access, decoding, cropping, validation and normalization, including their competition for CPU resources. The repeated pool is a diagnostic workload. A reusable cache for the full dataset would require its own integrity and throughput checks.

In [ ]:
rows = []
for name,label in [("steady_mvm_b16_io2", "Zarr input pipeline"), ("steady_mvm_cached_b16_io2", "32 normalized clips in shared RAM")]:
    r = read(name)
    rows.append({"Input": label, "Training clips/s": r["clips_per_second"],
                 "CPU cores in use, mean": r["resources"]["cpu_seconds"]/r["wall_seconds"]})
input_control = show_table(rows, {"Training clips/s": "{:.0f}", "CPU cores in use, mean": "{:.1f}"})

## Batch sizes in the complete pipeline

This comparison includes loading, host memory copies, GPU transfer, forward and backward passes, gradient checks, weight updates and scalar logging. Every configuration retains an effective batch of 32 clips. The I/O batch of 16 removes the explicit concatenation in the main process, while each worker prepares a larger group. Sustained throughput depends on both effects and on competition for CPU and memory resources. The two plots use identical vertical limits to support comparisons between objectives.

In [ ]:
rows = []
for o in ["mvm", "future"]:
    for batch,io in [(16,2),(16,16),(32,32)]:
        r = read(f"steady_{o}_b{batch}_io{io}")
        rows.append({"Objective": OBJECTIVES[o], "GPU batch": batch, "I/O batch": io, "Effective batch": r["effective_batch"],
                     "Clips/s": r["clips_per_second"], "GPU allocated (GiB)": r["peak_allocated_gib"],
                     "Wait (ms/batch)": r["batch_io"]["loader_wait_ms"]["mean_ms"]})
pipeline = show_table(rows, {"Clips/s": "{:.0f}", "GPU allocated (GiB)": "{:.1f}", "Wait (ms/batch)": "{:.1f}"})
fig,axes = plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax,o in zip(axes,["MVM","Future"]):
    f = pipeline[pipeline.Objective == o]
    ax.bar(["GPU 16 / I/O 2", "GPU 16 / I/O 16", "GPU 32 / I/O 32"],f["Clips/s"],color=["#526579","#16827c","#d8943a"])
    ax.set_title(o); ax.set_ylim(0,pipeline["Clips/s"].max()*1.15); ax.tick_params(axis="x",rotation=20)
axes[0].set_ylabel("Training clips/s")
fig.tight_layout(); fig.savefig(FIGURES / "pipeline_batch_sizes.png",bbox_inches="tight"); plt.show()


## Compute capacity and GPU memory

The next measurements keep the input tensor on the GPU. Each trial performs four warmup batches and eight timed training batches, except the initial batch-16 reference tests, whose configuration files retain their exact counts. Larger batches reuse a pool of up to 32 real clips. The resulting throughput estimates the capacity of the model execution path when data preparation and transfers have already been completed.

“Allocated memory” is the peak memory held by live PyTorch tensors, including the resident input. The CUDA allocator can reserve additional memory for reuse. GPU batches of 16 and 32 both retain an effective batch of 32. GPU batches of 64 and 128 also increase the effective batch to 64 and 128 respectively, so using them in a scientific run would change its optimization schedule. Joint at batch 128 exceeded the available GPU memory; its table entry records that outcome.

The legend names match the five objectives introduced at the start of this notebook. Throughput and memory are plotted separately because their units and interpretations differ.

In [ ]:
rows = []
for o,label in OBJECTIVES.items():
    for batch in [16,32,64,128]:
        name = f"gpu_{o}_b{batch}"
        if not (REVIEW / name / "result.json").exists():
            name = f"gpu_{o}_baseline"
        r = read(name)
        rows.append({"Objective": label, "GPU batch": batch, "Status": r["status"],
                     "Clips/s": r.get("clips_per_second",np.nan), "GPU allocated (GiB)": r.get("peak_allocated_gib",np.nan)})
gpu = show_table(rows, {"Clips/s": "{:.0f}", "GPU allocated (GiB)": "{:.1f}"})
fig,axes = plt.subplots(1,2,figsize=(11,4))
for label in OBJECTIVES.values():
    f = gpu[(gpu.Objective == label) & (gpu.Status == "completed")]
    axes[0].plot(f["GPU batch"],f["Clips/s"],marker="o",label=label)
    axes[1].plot(f["GPU batch"],f["GPU allocated (GiB)"],marker="o",label=label)
axes[0].set(title="GPU-resident training",ylabel="Training clips/s",ylim=(0,None))
axes[1].set(title="Peak allocated GPU memory",ylabel="GiB",ylim=(0,33))
for ax in axes:
    ax.set_xlabel("GPU batch size (clips)"); ax.set_xticks([16,32,64,128]); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / "gpu_batch_scaling.png",bbox_inches="tight"); plt.show()


## GPU activity and the missing utilization percentage

On this MIG/vGPU configuration, NVML returns `Not Supported` for utilization queries on both the exposed device and its MIG handle. `nvidia-smi` therefore displays `N/A`. NVIDIA documents this limitation in the [nvidia-smi reference](https://docs.nvidia.com/deploy/nvidia-smi/index.html). Nsight Systems also reports that this virtual GPU lacks permission for GPU hardware-counter profiling.

PyTorch's CUDA activity tracing is available. The table reports the fraction of the measured training interval covered by at least one GPU kernel, and the fraction covered by kernels, GPU memory copies or memory initialization combined. Overlapping events count once. These fractions measure activity over elapsed time. Multiprocessor occupancy describes a separate property of kernel execution and requires suitable hardware counters. Access to host-side DCGM profiling is an administrative option described in the [MIG monitoring documentation](https://docs.nvidia.com/datacenter/tesla/mig-user-guide/latest/getting-started-with-mig.html).

The traces use the MVM objective, a GPU batch of 16, eight workers, 32 warmup batches and 32 measured batches. Trace collection adds overhead, so throughput comparisons use the untraced runs above. The saved Chrome traces can be opened in a compatible trace viewer to inspect individual CPU calls, CUDA kernels and transfers.

In [ ]:
rows = []
for io in [2,16]:
    r = read(f"steady_trace_mvm_io{io}"); t = r["trace"]
    rows.append({"I/O batch": io, "Measured interval (s)": t["window_ms"]/1000,
                 "Kernel activity (%)": 100*t["kernel_active_fraction"],
                 "All device activity (%)": 100*t["device_activity_active_fraction"], "Traced clips/s": r["clips_per_second"]})
activity = show_table(rows, {"Measured interval (s)": "{:.1f}", "Kernel activity (%)": "{:.0f}",
                             "All device activity (%)": "{:.0f}", "Traced clips/s": "{:.0f}"})


## Validation operations

Validation evaluates all 512 held-out clips. The model forward pass is only one part of that calculation: the same pass also computes persistence, advection, errors by channel and forecast horizon, gradient errors and metrics within the three historical-variability groups. Accumulation is repeated for every available predictor and variability group. The table sums the elapsed main-process intervals for each named operation across the full validation sample.

The residual time includes loading, host-to-device transfers, construction of masks and selections, scalar conversions, final report construction and Python control flow. GPU execution is asynchronous, so synchronization can be charged to the next operation that requires its result. MVM produces its masked reconstruction loss; the forecast metric accumulation in its validation pass concerns persistence and advection. Future additionally contributes neural forecast metrics. Both passes were warmed through the complete evaluation path before measurement.

In [ ]:
labels = {"validation_advection": "Advection baseline", "validation_model_forward": "Model forward pass",
          "validation_variability": "Historical variability", "validation_metric_accumulation": "Metric accumulation",
          "validation_plot": "Forecast figure"}
rows = []
for o in ["mvm","future"]:
    r = read(f"steady_validation_{o}"); total = 0
    for key,label in labels.items():
        t = r["timings"]["host_intervals"][key]; seconds = t["mean_ms"]*t["samples"]/1000; total += seconds
        rows.append({"Objective": OBJECTIVES[o], "Operation": label, "Time (s/512 clips)": seconds})
    rows.append({"Objective": OBJECTIVES[o], "Operation": "Remaining validation operations", "Time (s/512 clips)": r["wall_seconds"]-total})
    rows.append({"Objective": OBJECTIVES[o], "Operation": "Total validation", "Time (s/512 clips)": r["wall_seconds"]})
validation = show_table(rows, {"Time (s/512 clips)": "{:.1f}"})


## Decoded frames and batch delivery

The follow-up experiment separates storage format from the path used to deliver a batch. Its reference repeats the original MVM pipeline. The first intervention replaces compressed Zarr reads with two daily NPY arrays containing the native multichannel frames in Kelvin float16. It retains the original clip reader's checks, orientation and CPU normalization. Each native frame is stored once and serves every overlapping crop that references it.

Preparing the two days took **47 seconds** and produced **10.1 GiB**. The builder compared the stored float16 bit patterns with every decoded source block, reread the files for checksums and verified that source files remained stable during construction. Separate checks compared 32 real clips with Zarr and verified another 64 clips after threaded batch preparation and CUDA transfer. Time order, channel order, spatial orientation and source temperatures were preserved exactly.

The subsequent interventions copy native float16 values directly into pinned host batches and apply the original global training normalization on the GPU. CPU and GPU normalization differ by less than **0.001 K** in equivalent temperature in the control sample. The complete 16-frame tensor is checked for finite values on the GPU before it is split into model history and forecast targets. Tests verify that an invalid clip is rejected before forecasting. The CPU version of this check remains available for independent cache reads.

The comparisons process 2,048 clips after 512 warmup clips, with unused inputs beyond the measured interval to keep producers active. Every training configuration uses an effective batch of 32 clips. The source training process was temporarily suspended with its state preserved while these measurements ran.

In [ ]:
cache_cases = [
    ("Zarr; original loader", "cache_mvm_zarr_baseline"),
    ("Decoded frames; original loader", "cache_mvm_npy_baseline"),
    ("Raw batches; 8 processes; CPU finite check", "cache_mvm_raw_process8"),
    ("Raw batches; 2 threads; CPU finite check", "cache_mvm_raw_threads2"),
    ("Raw batches; 2 threads; GPU finite check", "gpu_validate_mvm_b16"),
    ("Raw batches; 4 threads; GPU finite check", "threads4_mvm_b16"),
    ("Raw batches; 4 threads; GPU batch 32", "threads4_mvm_b32"),
]
rows = []
for label,name in cache_cases:
    r = read(name)
    rows.append({"MVM configuration": label, "GPU batch": r["batch_size"], "Clips/s": r["clips_per_second"],
                 "CPU cores, mean": r["resources"]["cpu_seconds"]/r["wall_seconds"],
                 "Allocated GPU memory (GiB)": r["peak_allocated_gib"]})
cache_comparison = show_table(rows, {"Clips/s": "{:.0f}", "CPU cores, mean": "{:.1f}", "Allocated GPU memory (GiB)": "{:.1f}"})
fig,ax = plt.subplots(figsize=(10,5))
ax.barh(cache_comparison["MVM configuration"],cache_comparison["Clips/s"],color="#16827c")
ax.invert_yaxis(); ax.set_xlabel("Training clips/s"); ax.set_title("MVM input pipeline")
fig.tight_layout(); fig.savefig(FIGURES / "decoded_frame_pipeline.png",bbox_inches="tight"); plt.show()


The file-format substitution provides a modest gain on its own. The larger improvement comes from reducing host copies and float16 CPU processing while retaining the complete validity check before model execution. In an isolated CPU timing, checking all values of one float16 clip took approximately 8 ms; the native-frame reader took approximately 15 ms with that check and 6 ms when validation was deferred to the GPU. These CPU microbenchmarks reuse one clip and quantify that operation under their own cache conditions.

The threaded loader prepares complete batches directly in pinned memory. Two threads with two pending batches and four threads with four pending batches were tested. Increasing the preparation parallelism helps MVM in this experiment. Future benefits less from that change because its full-history model consumes more GPU compute per clip. The following comparison reports the better measured settings for each objective; each point is a timing trial on this workstation.

In [ ]:
rows = []
for objective,baseline,candidate in [("MVM", "cache_mvm_zarr_baseline", "threads4_mvm_b32"),
                                    ("Future", "steady_future_b16_io2", "gpu_validate_future_b32")]:
    a,b = read(baseline),read(candidate)
    rows.append({"Objective": objective, "Original clips/s": a["clips_per_second"], "Candidate clips/s": b["clips_per_second"],
                 "Throughput ratio": b["clips_per_second"]/a["clips_per_second"], "Candidate GPU batch": b["batch_size"],
                 "Effective batch": b["effective_batch"]})
show_table(rows, {"Original clips/s": "{:.0f}", "Candidate clips/s": "{:.0f}", "Throughput ratio": "{:.1f}×"});


## GPU activity with reduced tracing overhead

Nsight Systems collected CUDA API and kernel activity between explicit measurement markers. CPU instruction sampling and GPU hardware-counter profiling were disabled. The available CUDA activity trace provides the kernel and memory-copy intervals used below. This capability complements the unsupported hardware-utilization queries described earlier.

The before/after MVM comparison uses the same Nsight collection method. Kernel activity is the fraction of elapsed time covered by a running kernel; total device activity also includes copies and memory initialization. The GPU-resident MVM case measures the model execution path with inputs already on the device. Its approximately 80% total activity establishes an attainable reference for this workload and batch size. Pipeline activity also reflects gaps introduced by data preparation, transfers and host-side scheduling. Hardware counters would be needed to measure how many multiprocessors are occupied during each active interval.

Throughput is compared using the untraced measurements above because collecting traces adds runtime overhead. All activity fractions below are calculated from the union of intervals within the marked range, so concurrent operations count once.

In [ ]:
rows = []
for label,name in [("MVM: original pipeline, batch 16", "nsys_baseline_mvm_b16"),
                   ("MVM: candidate pipeline, batch 32", "nsys_threads4_mvm_b32"),
                   ("Future: candidate pipeline, batch 32", "nsys_pipeline_future_b32"),
                   ("MVM: GPU-resident inputs, batch 32", "nsys_gpu_mvm_b32")]:
    a = json.loads((REVIEW/name/"activity_summary.json").read_text())
    rows.append({"Configuration": label, "Kernel activity (%)": a["kernel_percent"], "Total device activity (%)": a["all_device_percent"]})
nsight_activity = show_table(rows, {"Kernel activity (%)": "{:.0f}", "Total device activity (%)": "{:.0f}"})


## Adoption in the training series: complete clip files

The two training manifests contain 4,096 and 8,192 clips, with the smaller set fully included in the larger. Materialization stores one physical file per unique clip: 8,192 training files and 512 validation files. Each file contains 16 frames, seven channels and a 224 × 224 spatial crop in Kelvin, in the orientation and temporal order established by the original reader. LZ4 compression preserves every native float16 value. Preparation took approximately 2.5 minutes and produced 39.6 GiB; every encoded clip was decoded and compared with its source, and the saved file bytes were checked before publication. A checksum is checked on every subsequent read.

Four threads decode complete batches into pinned host memory. Transfer uses float16, followed by float32 conversion and the stored global training normalization on the GPU. The complete input is checked for finite values before model execution. Both training and validation use this path. An epoch contains 128 training batches for 4,096 clips and 256 for 8,192; validation contains 16 batches for its 512 clips. Each training batch contains 32 clips and produces one optimizer update. The original configuration used twice as many training batches, with accumulation over two batches for each update.

The following table compares the previous training logs with complete epochs using the adopted pipeline. The first epoch in each timing run includes initialization and is excluded from the steady comparison. Epochs recorded with Nsight are also excluded from elapsed-time averages. The MVM 8,192 measurement resumes the scientific checkpoint at epoch 8; the other timing runs use isolated model instances. Checkpoint and log persistence are included in the new full-epoch duration. The original full-epoch column sums training and validation because the historical logs record those components separately.

CUDA activity is measured over a complete training epoch with Nsight Systems, using the same event-union method as the preceding diagnostic measurements. Kernel activity is the fraction of elapsed time with at least one kernel running. Total device activity also includes transfers and memory initialization. These percentages refer to the allocated GPU partition and measure activity over time.


In [ ]:
REVISION = ROOT / "results/ssl_scaling_20261025/execution_revisions/clip_files_v1"
comparison_path = REVISION / "comparison.json"
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text())
    show_table(comparison["timing_rows"], {key: "{:.0f}" for key in comparison["timing_format_columns"]})
    show_table(comparison["activity_rows"], {"Kernel activity (%)": "{:.0f}", "Total device activity (%)": "{:.0f}"})
    show_table(comparison["batch_rows"])
    show_table(comparison["memory_rows"], {"Previous allocated GPU (GiB)": "{:.1f}", "Current allocated GPU (GiB)": "{:.1f}"})
else:
    print("Complete-epoch measurements are being collected.")


## Scope of the result

Direct preparation of pinned batches and global normalization on the GPU improve the measured input throughput. The adopted storage uses complete Kelvin clips compressed without loss. The file collection occupies approximately 40 GiB and fits the available filesystem cache; larger collections and other storage systems require a sustained throughput measurement under their own conditions.

The execution revision records the clip collection, loader settings, normalization device, source hashes and checkpoint lineage. Effective batch size, learning rate, model architecture, losses, training observations and validation observations retain their registered values. Floating-point reduction order and stochastic mask grouping can vary when two batches of 16 become one batch of 32. The saved revision identifies the point of this change in each resumed training history.


## Reproducing the measurements

The standalone script `scripts/profile_seviri_ssl.py` writes the configuration, operation timings, resource counters and optional CUDA trace to a separate result directory. Use an idle GPU for a comparable result. A steady baseline measurement can be launched from the repository root with:

```bash
OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 NUMEXPR_NUM_THREADS=1 \
.venv/bin/python scripts/profile_seviri_ssl.py \
  --dataset results/ssl_scaling_20261025/n4096/dataset \
  --output-dir results/performance_new_measurement \
  --mode pipeline --objective mvm \
  --batch-size 16 --accumulation 2 --io-batch 2 --workers 8 \
  --warmup 32 --steps 128
```

`--trace` enables CUDA activity recording. `--mode gpu` measures training with resident GPU inputs; `--mode io` measures data preparation; `--mode validation --workers 4 --steps 32` evaluates 512 clips at batch size 16. Each invocation requires a new output directory. The source notebook contains the analysis code, and its executed snapshot and figures are stored with the benchmark results.

To reproduce the decoded-frame candidate, create a new cache directory and profile it separately:

```bash
OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 \
.venv/bin/python scripts/build_seviri_frame_cache.py \
  --zarr-root /home/isac/deep_convection_data/MSG-SEVIRI \
  --manifest results/ssl_scaling_20261025/n4096/dataset/train.csv \
  --output-dir /home/isac/deep_convection_data/cache_native_seviri_new

OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 NUMEXPR_NUM_THREADS=1 \
.venv/bin/python scripts/profile_seviri_ssl.py \
  --dataset results/ssl_scaling_20261025/n4096/dataset \
  --frame-cache /home/isac/deep_convection_data/cache_native_seviri_new \
  --output-dir results/performance_new_cache_measurement \
  --mode pipeline --objective mvm --input-mode raw --finite-on-gpu \
  --loader-kind thread --workers 4 --thread-prefetch 4 \
  --batch-size 32 --accumulation 1 --io-batch 32 --warmup 16 --steps 64
```
